# Dev check: SuddenNetZeroPulse coupler flux pulse vs. qubit operations

**Not a numbered calibration node.** This notebook is a Phase 0 mechanism check before
any CZ-gate chevron node is built: does a schedule-internal `SuddenNetZeroPulse` (the
standard SNZ conditional-phase-gate flux shape) run correctly on a **coupler's** flux
port in the same schedule as reset+X90+Measure on its two adjacent qubits, and does the
coupler's flux output return to its externally applied idle bias once the pulse ends?

The schedule is **sequential**: reset+X90 on both qubits (in parallel with each other),
then the coupler pulse, then measure on both qubits (in parallel with each other) — not a
literal overlap of the flux pulse with the drive pulses. This isn't just a simplification:
the installed `qblox_scheduler` version's timing validator fails whenever a baseband/flux
pulse (`VoltageOffset`/`SuddenNetZeroPulse`) is aligned via `ref_op`, confirmed by direct
testing. See `dev_snz_flux_pulse_check.py`'s module docstring for details.

This does **not** validate any physics (no chevron sweep, no population/frequency
analysis) — there is no existing coupler-flux ground truth to compare against yet.
It only proves the scheduling/hardware mechanism works. Start with small `AMP_A`/
`T_PULSE` values well inside a range you trust is safe for this coupler.

In [1]:
from pathlib import Path

from qblox_lab.config.hardware import create_hardware_agent
from qblox_lab.config.sessions import CONFIG_DIR, SESSIONS
from qblox_lab.experiments.dev_snz_flux_pulse_check import SNZFluxPulseCheck

## Run parameters

In [9]:
SESSION = SESSIONS["AS_QRC"]
HARDWARE_CONFIG = SESSION.hardware_config
DEVICE_CONFIG = SESSION.device_config
FLUX_CONFIG = SESSION.flux_config  # Idle bias applied to both qubits and the coupler
OUTPUT_DIR = Path("data")

QUBIT_A = "q1"
QUBIT_B = "q2"
COUPLER_PORT = "cpl1_2:fl"     # Hardware connectivity port, wired in hw_config_AS_QRC.json
COUPLER_FLUX_KEY = "cpl1_2"    # Key in flux_config.json's flux_biases (has an explicit "port")

REPETITIONS = 5

# SuddenNetZeroPulse shape parameters (Negirneac 2021). Conservative first pass:
# small amplitude, short duration, no correction segment yet.
AMP_A = 0.1
AMP_B = 0.0
NET_ZERO_A_SCALE = 1.0
T_PULSE = 100e-9
T_PHI = 0.0
T_INTEGRAL_CORRECTION = 4e-9

IDLE_SETTLE_TIME = 1e-6
IDLE_RETURN_TIME = 1e-6

TIMEOUT = 300
CREATE_DUMMY_CONNECTIONS = False

## Hardware and experiment

In [10]:
import atexit

from qcodes.instrument import Instrument

hardware_agent = create_hardware_agent(
    hardware_configuration=HARDWARE_CONFIG,
    device_configuration=DEVICE_CONFIG,
    output_dir=OUTPUT_DIR,
    create_dummy_connections=CREATE_DUMMY_CONNECTIONS,
)

experiment = SNZFluxPulseCheck(
    hardware_agent=hardware_agent,
    qubit_a=QUBIT_A,
    qubit_b=QUBIT_B,
    coupler_port=COUPLER_PORT,
    coupler_flux_key=COUPLER_FLUX_KEY,
    flux_config=FLUX_CONFIG,
)

atexit.register(Instrument.close_all)

<bound method Instrument.close_all of <class 'qcodes.instrument.instrument.Instrument'>>

## Measurement

In [11]:
dataset = experiment.run_measurement(
    repetitions=REPETITIONS,
    amp_A=AMP_A,
    amp_B=AMP_B,
    net_zero_A_scale=NET_ZERO_A_SCALE,
    t_pulse=T_PULSE,
    t_phi=T_PHI,
    t_integral_correction=T_INTEGRAL_CORRECTION,
    idle_settle_time=IDLE_SETTLE_TIME,
    idle_return_time=IDLE_RETURN_TIME,
    timeout=TIMEOUT,
)

print(f"Coupler flux bias before run: {dataset.attrs['coupler_flux_bias_before']:.6f} V")
print(f"Coupler flux bias after run:  {dataset.attrs['coupler_flux_bias_after']:.6f} V")
dataset

/home/reny871224/micromamba/envs/qblox-training/lib/python3.10/site-packages/qblox_scheduler/backends/qblox/compiler_abc.py:285: UserWarning: Exact capabilities for ISA version 2.1 not found, using capabilities for version 2.2.
  warnings.warn(


Coupler flux bias before run: 0.000005 V
Coupler flux bias after run:  0.000005 V


<xarray.Dataset> Size: 48B
Dimensions:           (acq_index_S21_q1: 1, acq_index_S21_q2: 1)
Coordinates:
  * acq_index_S21_q2  (acq_index_S21_q2) int64 8B 0
  * acq_index_S21_q1  (acq_index_S21_q1) int64 8B 0
Data variables:
    S21_q1            (acq_index_S21_q1) complex128 16B (-0.00135973613262176...
    S21_q2            (acq_index_S21_q2) complex128 16B (-0.00144314265251159...
Attributes:
    tuid:                      20260828-065028-074-1ec9b6
    coupler_flux_bias_before:  4.76837158203125e-06
    coupler_flux_bias_after:   4.76837158203125e-06

## Optional: view the compiled pulse schedule

This check's schedule is already small (two qubits x a handful of operations per
repetition), so the schedule from the real run above is safe to plot directly —
no separate dummy schedule needed. Look for the SNZ pulse's two-lobe shape on the
coupler's flux port, playing after both qubits' X90 pulses finish and before the
readout pulses.

In [12]:
compiled_schedule = hardware_agent.latest_compiled_schedule
if compiled_schedule is None:
    raise RuntimeError("Run experiment.run_measurement() first.")
compiled_schedule.plot_pulse_diagram(plot_backend="plotly")

## Next step

If the coupler flux bias after the run matches the bias before it (within the ramp
parameter's tolerance), and the pulse diagram shows the expected two-lobe SNZ shape on
the coupler's flux port, the SNZ-on-coupler mechanism is validated as a prerequisite for
building a real CZ-gate chevron node (which will need to place the flux pulse to overlap
the relevant window in time, not just run after the prep gates as this check does).

## Release hardware connection

In [ ]:
from qcodes.instrument import Instrument

Instrument.close_all()